# 01 Collection check

Sanity checks on the raw reviews collected in Phase 1. All logic lives in `src/collect/`; this notebook only calls it.

Run the collectors first:
```
python -m src.collect.play_store
python -m src.collect.app_store
```

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.collect.summary import load_raw, collection_summary
from src.config import set_seed
set_seed()

df = load_raw()
df.shape

In [ ]:
collection_summary(df)

## Privacy check
No reviewer names, images or user IDs should exist in any raw file.

In [ ]:
forbidden = [c for c in df.columns if any(k in c.lower() for k in ('user', 'name', 'image', 'author'))]
assert not forbidden, forbidden
print('OK: no personal columns ->', list(df.columns))

## Schema and duplicates

In [ ]:
print(df.dtypes)
print('duplicate ids per source:', df.duplicated(['source', 'review_id']).sum())

## Monthly volume (quick look; full EDA is in 02_eda)

In [ ]:
play = df[df['source'] == 'google_play']
monthly = play.groupby([play['review_date'].dt.to_period('M'), 'app']).size().unstack(fill_value=0)
monthly.tail(24)

In [ ]:
df.sample(5, random_state=42)[['app', 'source', 'lang_query', 'rating', 'review_date', 'text']]